# inplace-op-unsafe-warning — ex2: inplace_unsafe context manager that toggles the guard, restores on exit

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inplace-op-unsafe-warning`. Running the final beacon cell reports progress against the `Backprop: In-place op unsafe warning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: In-place op unsafe warning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-op-unsafe-warning`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-op-unsafe-warning"
DD_SUBTOPIC = "Backprop: In-place op unsafe warning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## In-place guard — context-manager escape hatch — quick refresher

Ex1's `add_inplace_safe` ALWAYS refuses when `.recipe is not None`. But advanced users sometimes need to override (e.g. they're explicitly detaching, or they know the cached value is dead). The canonical pattern is a context manager that toggles a module-level bool:

```python
with inplace_unsafe():
    add_inplace_safe(x, y)   # would normally refuse — now allowed
# guard automatically re-armed here
```

Implementation pattern: save the old value of a module-level flag, set it to False (or True, depending on convention), yield, restore the old value in `finally` (so even an exception inside the `with` block doesn't leave the guard disarmed).

### Exercise 2 — inplace_unsafe context manager that toggles the guard, restores on exit

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the context-manager guard-toggle pattern: save the prior flag value, override for the block, restore on exit using a try/finally so exceptions don't leave the guard disabled.
> Keywords: context-manager, guard, toggle, in-place, finally
> ```

**KCs targeted:** `inplace-op-unsafe-warning`, `recipe-dataclass`

We've set up a module-level flag `_INPLACE_GUARD_ARMED = True` and a modified `add_inplace_safe(x, y)` that consults it: it refuses if `x.recipe is not None` AND `_INPLACE_GUARD_ARMED` is True; otherwise it mutates.

Implement `inplace_unsafe()` — a CONTEXT MANAGER (use `@contextlib.contextmanager`) that:

1. **Saves the prior value** of `_INPLACE_GUARD_ARMED`.
2. **Sets `_INPLACE_GUARD_ARMED = False`** for the duration of the `with` block (i.e. disables the refusal).
3. **Yields** (the `with` body runs).
4. **Restores the prior value** in a `finally` block, so even an exception inside the `with` body re-arms the guard.

Critical: the restore MUST be in `finally`, not after the yield. If the body raises, code after `yield` is skipped — but `finally` always runs.

Tests verify three things:
- Inside the block, `add_inplace_safe` MUTATES a recipe-carrying tensor (no refusal).
- Outside the block, `add_inplace_safe` REFUSES again.
- An exception inside the block still re-arms the guard.

Note: the global is named `_INPLACE_GUARD_ARMED` — use the `global` statement to mutate it from inside the context manager.

In [ ]:
import contextlib

@contextlib.contextmanager
def inplace_unsafe():
    global _INPLACE_GUARD_ARMED
    prev = _INPLACE_GUARD_ARMED
    _INPLACE_GUARD_ARMED = False
    try:
        yield
    finally:
        # finally runs even on exception — guard is always restored.
        _INPLACE_GUARD_ARMED = prev


<details><summary>Solution</summary>

```python
import contextlib

@contextlib.contextmanager
def inplace_unsafe():
    global _INPLACE_GUARD_ARMED
    prev = _INPLACE_GUARD_ARMED
    _INPLACE_GUARD_ARMED = False
    try:
        yield
    finally:
        # finally runs even on exception — guard is always restored.
        _INPLACE_GUARD_ARMED = prev
```

**Why `try/finally` is non-negotiable.** Code AFTER the `yield` in a context manager runs only if the body completes normally. If the body raises, that code is skipped — and the guard stays disabled forever. `finally` runs unconditionally, even on exception, even on `KeyboardInterrupt`. This is how PyTorch's `torch.no_grad()`, `torch.enable_grad()`, and `torch.set_grad_enabled()` are all written.

**Why save `prev`, not hard-code `True`.** Nested `with` blocks. If you hard-code restoring to True, then an outer `with inplace_unsafe():` followed by an inner `with inplace_unsafe():` would have the inner's exit re-arm the guard — but the outer expected it to stay disabled. Saving the previous value composes correctly under nesting.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()